# FastballAPI: a season of baseball in five minutes

This notebook loads the free FastballAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: FastballAPI (fastballapi.com), under the [FastballAPI Sample Data Licence](https://github.com/sports-data-hq/fastballapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/fastballapi-mcp/releases/download/season-pack/fastballapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('mlb', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## The final standings

In [ ]:
t('mlb', 'standings').head(10)

## Pitchers by run value saved, minimum 100 innings

In [ ]:
p = t('mlb', 'pitching')
p[p.innings_pitched >= 100].sort_values('run_value_saved', ascending=False)[['pitcher', 'team', 'innings_pitched', 'so', 'bb', 'ra9', 'run_value_saved']].head(10)

## Teams by our end-of-season Elo

In [ ]:
t('mlb', 'teams').sort_values('elo_end_of_season', ascending=False)[['team', 'elo_end_of_season', 'home_runs', 'batting_run_value']].head(10)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the FastballAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=fastballapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/mlb/get_standings', data=b'{}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the baseball skills from https://github.com/sports-data-hq/fastballapi-mcp.